# 🚀 Project 12 · vLLM Serving Experiments

Run Extensions 2–5 on one Colab GPU. Use a fresh T4 runtime so the baseline notebook's
server and these experiments do not compete for port 8000 or GPU memory.

Run the setup cell once, then run each experiment cell separately. Keep the runtime
open until you download the results ZIP. Each experiment starts and stops its own
loopback server.

In [ ]:
!nvidia-smi

In [ ]:
%%bash
set -e
if [ ! -d /content/vllm-agent-inference/.git ]; then
  git clone https://github.com/anne500/vllm-agent-inference.git /content/vllm-agent-inference
fi
git -C /content/vllm-agent-inference rev-parse HEAD
python -m pip install -q uv
uv venv --python 3.12 /content/vllm-env
uv pip install --python /content/vllm-env/bin/python \
  --torch-backend=cu129 "vllm==0.12.0" ninja

## 📊 Extension 2 · Repeatable streaming benchmark

Run three trials at each concurrency level. Capture client-side time to first text and total latency.

In [ ]:
%%bash
/content/vllm-env/bin/python /content/vllm-agent-inference/run_extensions.py streaming \
  --output /content/project12-extension-results

## 🟣 Extension 3 · Prefix caching comparison

Start a fresh server for each cache setting. Replay identical requests and record cache counters.

In [ ]:
%%bash
/content/vllm-env/bin/python /content/vllm-agent-inference/run_extensions.py prefix \
  --output /content/project12-extension-results

## 🚦 Extension 4 · Capacity and queue monitoring

Schedule requests at four arrival rates and sample the waiting and running request gauges.

In [ ]:
%%bash
/content/vllm-env/bin/python /content/vllm-agent-inference/run_extensions.py capacity \
  --output /content/project12-extension-results

## 🟪 Extension 5 · BRI workflow benchmark

Extract fields and draft a triage sentence from synthetic CI logs. Record timings and deterministic field checks.

In [ ]:
%%bash
/content/vllm-env/bin/python /content/vllm-agent-inference/run_extensions.py bri \
  --output /content/project12-extension-results

## 🗂️ Export measured results

Run after the four experiments finish. This archive includes JSON results and vLLM startup logs.

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

results = Path("/content/project12-extension-results")
expected = {"streaming.json", "prefix.json", "capacity.json", "bri.json"}
missing = sorted(expected - {path.name for path in results.glob("*.json")})
if missing:
    print("Still missing:", ", ".join(missing))
if any(results.glob("*.json")):
    archive = shutil.make_archive("/content/project12-extension-results", "zip", results)
    files.download(archive)